In [ ]:
# ==============================================================================
# 🛠️ Étape 1 : Installations et Initialisation
# ==============================================================================
!pip install -q datasets transformers evaluate accelerate matplotlib seaborn torch

import os
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, Trainer, TrainingArguments
import evaluate

# Fixer la graine pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

# ==============================================================================
# 📊 Étape 2 : Chargement des Données & Inspection (Task 1)
# ==============================================================================
print("=== Étape 1 : Chargement et Inspection du Dataset ===")
# Chargement du dataset tweet_eval (configuration sentiment : 0=Négatif, 1=Neutre, 2=Positif)
dataset = load_dataset("tweet_eval", "sentiment")
print(dataset)

# Distribution des classes sur le train set
labels_train = dataset["train"]["label"]
unique, counts = np.unique(labels_train, return_counts=True)
print("\nDistribution des classes (0=Neg, 1=Neu, 2=Pos) :", dict(zip(unique, counts)))

# Sauvegarde de deux exemples par label pour la future inspection d'attention
example_tweets = {0: [], 1: [], 2: []}
for item in dataset["train"]:
    lbl = item["label"]
    if len(example_tweets[lbl]) < 2:
        example_tweets[lbl].append(item["text"])
    if all(len(v) == 2 for v in example_tweets.values()):
        break

print("\nExemples sauvegardés pour l'inspection :")
for lbl, tweets in example_tweets.items():
    print(f"Label {lbl}: {tweets[0][:80]}...")

# ==============================================================================
# ⚙️ Étape 3 : Pipeline de Tokenisation (Task 2)
# ==============================================================================
print("\n=== Étape 2 : Configuration du Tokenizer ===")
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Application du preprocessing
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

# ==============================================================================
# 🏋️ Étape 4 : Fine-Tuning Setup & Entraînement (Task 3)
# ==============================================================================
print("\n=== Étape 3 : Configuration du Modèle et Entraînement ===")
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)

# Définition des métriques
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1_macro}

training_args = TrainingArguments(
    output_dir="./distilbert_sentiment_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# Lancement du Fine-Tuning
trainer.train()

# Sauvegarde finale du meilleur checkpoint
trainer.save_model("./best_trustworthy_distilbert")
tokenizer.save_pretrained("./best_trustworthy_distilbert")
print("✅ Meilleur modèle sauvegardé dans './best_trustworthy_distilbert'")

# ==============================================================================
# 📉 Étape 5 : Évaluation & Histogramme de Calibration (Task 4)
# ==============================================================================
print("\n=== Étape 4 : Évaluation & Histogramme de Calibration ===")
# Évaluation sur la validation
val_results = trainer.evaluate(tokenized_datasets["validation"])
print(f"Validation Metrics : Accuracy = {val_results['eval_accuracy']:.4f}, F1 Macro = {val_results['eval_accuracy']:.4f}")

# Prédictions sur le test split pour récupérer les scores softmax
test_preds = trainer.predict(tokenized_datasets["test"])
logits = torch.tensor(test_preds.predictions)
softmax_scores = torch.nn.functional.softmax(logits, dim=-1)

# Extraire la confiance associée à la classe prédite
confidences, predicted_classes = torch.max(softmax_scores, dim=-1)
confidences = confidences.numpy()

# Plot de l'histogramme de confiance
plt.figure(figsize=(8, 5))
sns.histplot(confidences, bins=10, kde=True, color="skyblue", edgecolor="black")
plt.title("Histogramme de calibration des scores de confiance (Softmax)")
plt.xlabel("Score de Confiance (Max Softmax Probability)")
plt.ylabel("Nombre de Tweets")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("\n💡 Note sur la calibration : Si l'écrasante majorité des scores se situe au-dessus de 0.90 alors que l'accuracy globale est plus basse, le modèle souffre d'un problème de sur-confiance classique des architectures Transformers non calibrées.")

# ==============================================================================
# 👁️ Étape 6 : Inspection des Cartes d'Attention (Task 5)
# ==============================================================================
print("\n=== Étape 5 : Inspection de l'Attention Multitête ===")
# Chargement du modèle de base configuré pour renvoyer les poids d'attention
base_model = AutoModel.from_pretrained("./best_trustworthy_distilbert", output_attentions=True)
base_model.eval()

# Sélection d'un tweet d'exemple négatif sauvegardé plus tôt
test_tweet = example_tweets[0][0] # Premier tweet négatif
inputs = tokenizer(test_tweet, return_tensors="pt")

with torch.no_grad():
    outputs = base_model(**inputs)

# Attentions de la dernière couche : Shape [batch_size, num_heads, seq_len, seq_len]
last_layer_attention = outputs.attentions[-1][0] 

# Moyenne sur l'ensemble des têtes d'attention
avg_attention = torch.mean(last_layer_attention, dim=0)

# Extraction des poids dirigés depuis le token [CLS] (index 0) vers tous les autres tokens
cls_attention = avg_attention[0].numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Visualisation sous forme de graphique à barres pour la lisibilité textuelle
plt.figure(figsize=(12, 4))
sns.barplot(x=tokens, y=cls_attention, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title(f"Poids d'attention attribués par le token [CLS] (Texte Négatif)")
plt.ylabel("Intensité de l'attention moyenne")
plt.tight_layout()
plt.show()

# ==============================================================================
# 🔮 Étape 7 : Assistant d'Inférence Explicable Intégré
# ==============================================================================
print("\n=== Étape 6 : Déploiement de la fonction de Production analyze_text() ===")

def analyze_text(text: str, top_n_evidence: int = 3):
    # Encodage
    inputs = tokenizer(text, return_tensors="pt")
    
    # 1. Pipeline de Classification
    model.eval()
    with torch.no_grad():
        cls_outputs = model(**inputs)
    probs = torch.nn.functional.softmax(cls_outputs.logits, dim=-1)[0]
    conf, pred_idx = torch.max(probs, dim=-1)
    
    labels_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
    
    # 2. Pipeline d'Explicabilité (Attention)
    with torch.no_grad():
        base_outputs = base_model(**inputs)
    
    avg_attn = torch.mean(base_outputs.attentions[-1][0], dim=0)
    cls_attn_weights = avg_attn[0].numpy()
    tkns = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Isoler les tokens importants (en excluant les tokens spéciaux [CLS], [SEP], [PAD])
    evidence_indices = np.argsort(cls_attn_weights)[::-1]
    highlighted = []
    for idx in evidence_indices:
        token_str = tkns[idx]
        if token_str not in ["[CLS]", "[SEP]", "[PAD]"] and not token_str.startswith("##"):
            highlighted.append((token_str, float(cls_attn_weights[idx])))
        if len(highlighted) >= top_n_evidence:
            break
            
    return {
        "label": labels_map[int(pred_idx)],
        "confidence": float(conf),
        "highlighted_tokens": highlighted
    }

# Test concret pour simuler un outil de support
sample_support = "This product is an absolute nightmare, terrible customer service."
analysis = analyze_text(sample_support)
print(f"Texte analysé : '{sample_support}'")
print(f"Résultat renvoyé : {analysis}")